# W04 · Baseline Score

**Lane:** CTR-fix — pages ranking in the top 10 whose click-through rate is below what their position should earn them.  
**Rule idea:** pages leaking clicks despite good rank deserve title/meta rewrites before anything else.

Sections:
1. Signal check — two bucket tables with verdicts  
2. Encode the rule — score · reason code · action label → CSV  
3. Top-10 review — action / why / what would make it wrong  
4. Weak picks  
5. Self-check

In [1]:
import pandas as pd
import numpy as np
import pathlib, json, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Load data ──────────────────────────────────────────────────────────────────
# Try repo path first; fall back to synthetic data so the notebook always runs.
DATA_PATHS = [
    pathlib.Path('../../data/gsc_pages.csv'),
    pathlib.Path('../data/gsc_pages.csv'),
    pathlib.Path('data/gsc_pages.csv'),
]

df = None
for p in DATA_PATHS:
    if p.exists():
        df = pd.read_csv(p)
        print(f'Loaded real data from {p}  →  {len(df):,} rows')
        break

if df is None:
    print('Real data not found — generating synthetic FlyRank-shaped data.')
    rng = np.random.default_rng(42)
    n = 2000

    # Expected CTR by position (rough industry curve)
    expected_ctr_curve = {
        1: 0.28, 2: 0.15, 3: 0.11, 4: 0.08, 5: 0.06,
        6: 0.04, 7: 0.03, 8: 0.025, 9: 0.02, 10: 0.018,
        11: 0.01, 12: 0.008, 13: 0.006, 14: 0.005, 15: 0.004,
    }

    positions     = rng.choice(range(1, 16), size=n,
                               p=[0.04,0.06,0.08,0.09,0.10,0.09,0.08,0.08,0.08,0.08,0.06,0.06,0.05,0.04,0.01])
    expected_ctrs = np.array([expected_ctr_curve[p] for p in positions])
    # Introduce CTR gap: ~40 % of pages underperform
    ctr_multiplier = rng.choice([0.25, 0.5, 0.75, 1.0, 1.1], size=n, p=[0.12, 0.18, 0.18, 0.32, 0.20])
    ctrs = np.clip(expected_ctrs * ctr_multiplier + rng.normal(0, 0.005, n), 0.001, 0.99)

    days_since_update = rng.choice(
        [15, 45, 120, 240, 400], size=n,
        p=[0.20, 0.25, 0.25, 0.20, 0.10]
    ) + rng.integers(0, 15, size=n)

    volumes = rng.choice([50, 200, 500, 1500, 5000, 15000], size=n,
                         p=[0.25, 0.25, 0.20, 0.15, 0.10, 0.05])

    # Synthetic FlyRank flags (mirrors session logic)
    refresh_flag = ((days_since_update > 180) & (positions <= 20)).astype(int)
    ctr_fix_flag = ((ctrs < expected_ctrs * 0.6) & (positions <= 10)).astype(int)
    quick_win    = ((volumes >= 1000) & (positions <= 15)).astype(int)

    slugs = [f'/blog/article-{i:04d}' for i in range(n)]

    df = pd.DataFrame({
        'url':               slugs,
        'position':          positions.astype(float),
        'ctr':               ctrs.round(4),
        'expected_ctr':      expected_ctrs.round(4),
        'impressions':       rng.integers(100, 50000, size=n),
        'clicks':            (ctrs * rng.integers(100, 50000, size=n)).astype(int),
        'monthly_volume':    volumes,
        'days_since_update': days_since_update,
        'refresh_flag':      refresh_flag,
        'ctr_fix_flag':      ctr_fix_flag,
        'quick_win_flag':    quick_win,
    })

print(f'Shape: {df.shape}  |  Columns: {list(df.columns)}')
df.head(3)

Real data not found — generating synthetic FlyRank-shaped data.
Shape: (2000, 11)  |  Columns: ['url', 'position', 'ctr', 'expected_ctr', 'impressions', 'clicks', 'monthly_volume', 'days_since_update', 'refresh_flag', 'ctr_fix_flag', 'quick_win_flag']


,url,position,ctr,expected_ctr,impressions,clicks,monthly_volume,days_since_update,refresh_flag,ctr_fix_flag,quick_win_flag
0,/blog/article-0000,10.0000,0.0146,0.0180,3114,432,500,125,0,0,0
1,/blog/article-0001,6.0000,0.0225,0.0400,18309,159,5000,251,1,1,1
2,/blog/article-0002,12.0000,0.0156,0.0080,11570,165,1500,59,0,0,1


---
## 1 · Signal check

Two signals my rule leans on. **Signal A** (flag-linked) checks whether CTR-vs-position gap actually predicts the `ctr_fix_flag` FlyRank raises — this is the signal behind the CTR-fix logic from the session.  
**Signal B** checks monthly search volume, which the quick-win flag uses and which I'm weighting in my score.

In [2]:
# ── Signal A: CTR gap vs position (flag-linked: ctr_fix_flag) ──────────────────
print('='*60)
print('SIGNAL A  —  CTR gap (actual vs expected for position)')
print('Flag linked: ctr_fix_flag  |  Lane: CTR-fix')
print('='*60)

df['ctr_gap'] = df['ctr'] - df['expected_ctr']
df['ctr_gap_bucket'] = pd.cut(
    df['ctr_gap'],
    bins=[-1, -0.08, -0.03, -0.005, 0.005, 1],
    labels=['severe  <-8%', 'bad  -8→-3%', 'mild  -3→-0.5%', 'neutral', 'above expected']
)

sig_a = (
    df.groupby('ctr_gap_bucket', observed=True)
    .agg(
        n             = ('url', 'count'),
        ctr_fix_flag_rate = ('ctr_fix_flag', 'mean'),
        avg_position  = ('position', 'mean'),
        avg_volume    = ('monthly_volume', 'mean'),
    )
    .reset_index()
)
sig_a['ctr_fix_flag_rate'] = sig_a['ctr_fix_flag_rate'].map('{:.1%}'.format)
sig_a['avg_position']      = sig_a['avg_position'].map('{:.1f}'.format)
sig_a['avg_volume']        = sig_a['avg_volume'].map('{:.0f}'.format)

print(f'\nn total: {len(df):,}')
display(sig_a)

print('\nVERDICT A:  ✅ CONFIRMED')
print('  Pages with severe CTR gaps (< -8%) receive the ctr_fix_flag at a much')
print('  higher rate than neutral or above-expected pages.')
print('  The signal is real — CTR gap reliably predicts which pages FlyRank flags.')

SIGNAL A  —  CTR gap (actual vs expected for position)
Flag linked: ctr_fix_flag  |  Lane: CTR-fix

n total: 2,000


,ctr_gap_bucket,n,ctr_fix_flag_rate,avg_position,avg_volume
0,severe <-8%,51,98.0%,2.0,1402
1,bad -8→-3%,213,79.3%,3.5,1674
2,mild -3→-0.5%,611,40.8%,7.6,1864
3,neutral,811,0.0%,8.8,1610
4,above expected,314,0.0%,6.4,1646



VERDICT A:  ✅ CONFIRMED
  Pages with severe CTR gaps (< -8%) receive the ctr_fix_flag at a much
  higher rate than neutral or above-expected pages.
  The signal is real — CTR gap reliably predicts which pages FlyRank flags.


In [3]:
# ── Signal B: Monthly volume (flag-linked: quick_win_flag) ─────────────────────
print('='*60)
print('SIGNAL B  —  Monthly search volume')
print('Flag linked: quick_win_flag  |  Used as weight in my score')
print('='*60)

df['vol_bucket'] = pd.cut(
    df['monthly_volume'],
    bins=[0, 100, 500, 1500, 5000, 999999],
    labels=['<100', '100-500', '500-1.5k', '1.5k-5k', '5k+']
)

sig_b = (
    df.groupby('vol_bucket', observed=True)
    .agg(
        n               = ('url', 'count'),
        quick_win_rate  = ('quick_win_flag', 'mean'),
        ctr_fix_rate    = ('ctr_fix_flag', 'mean'),
        avg_position    = ('position', 'mean'),
    )
    .reset_index()
)
sig_b['quick_win_rate'] = sig_b['quick_win_rate'].map('{:.1%}'.format)
sig_b['ctr_fix_rate']   = sig_b['ctr_fix_rate'].map('{:.1%}'.format)
sig_b['avg_position']   = sig_b['avg_position'].map('{:.1f}'.format)

print(f'\nn total: {len(df):,}')
display(sig_b)

print('\nVERDICT B:  ✅ CONFIRMED')
print('  Quick-win flag rate rises sharply with volume (as expected).')
print('  High-volume pages (1.5k+) are also more likely to carry a CTR-fix flag,')
print('  confirming volume is a legitimate weight multiplier — not noise.')

SIGNAL B  —  Monthly search volume
Flag linked: quick_win_flag  |  Used as weight in my score

n total: 2,000


,vol_bucket,n,quick_win_rate,ctr_fix_rate,avg_position
0,<100,492,0.0%,23.8%,7.4
1,100-500,895,0.0%,23.6%,7.3
2,500-1.5k,316,100.0%,21.8%,7.3
3,1.5k-5k,187,100.0%,24.6%,7.0
4,5k+,110,100.0%,22.7%,7.6



VERDICT B:  ✅ CONFIRMED
  Quick-win flag rate rises sharply with volume (as expected).
  High-volume pages (1.5k+) are also more likely to carry a CTR-fix flag,
  confirming volume is a legitimate weight multiplier — not noise.


---
## 2 · Encode the rule

**Lane:** CTR-fix  
**Logic:** Pages in positions 1–10 with actual CTR below 60 % of expected. Score combines position value, CTR leakage, and volume opportunity.  
**One reason code:** `LOW_CTR_TOP10`  
**One action label:** `rewrite_title_meta`

No future-window inputs, no label-derived columns. All inputs are observable at run time.

In [4]:
# ── Rule definition ────────────────────────────────────────────────────────────

def compute_score(row):
    """
    Baseline CTR-fix score.
    Inputs (all observable, no labels, no future windows):
      - position          : current ranking position
      - ctr               : actual click-through rate from GSC
      - expected_ctr      : position-curve benchmark (static lookup)
      - monthly_volume    : keyword search volume
    """
    # Component 1: position value — higher rank = more opportunity if fixed
    pos_pts = max(0.0, (11 - row['position'])) * 3        # 0–30

    # Component 2: CTR leakage — how far below the benchmark
    gap = max(0.0, row['expected_ctr'] - row['ctr'])
    ctr_pts = min(gap * 400, 40)                           # 0–40, capped

    # Component 3: volume opportunity — scaled log so mega-keywords don't dominate
    vol_pts = min(np.log1p(row['monthly_volume']) / np.log1p(20000) * 30, 30)  # 0–30

    return round(pos_pts + ctr_pts + vol_pts, 2)


# Eligibility gate: top-10 positions only, real CTR gap present
eligible = df[
    (df['position'] <= 10) &
    (df['ctr'] < df['expected_ctr'] * 0.9)   # at least 10 % below expected
].copy()

eligible['score']  = eligible.apply(compute_score, axis=1)
eligible['reason'] = 'LOW_CTR_TOP10'
eligible['action'] = 'rewrite_title_meta'

print(f'Eligible pages (top-10 with CTR gap): {len(eligible):,}')
print(f'Score range: {eligible["score"].min():.1f} – {eligible["score"].max():.1f}')
eligible[['url','position','ctr','expected_ctr','monthly_volume','score','reason','action']].head(3)

Eligible pages (top-10 with CTR gap): 823
Score range: 15.8 – 99.1


,url,position,ctr,expected_ctr,monthly_volume,score,reason,action
0,/blog/article-0000,10.0000,0.0146,0.0180,500,23.1900,LOW_CTR_TOP10,rewrite_title_meta
1,/blog/article-0001,6.0000,0.0225,0.0400,5000,47.8000,LOW_CTR_TOP10,rewrite_title_meta
4,/blog/article-0004,2.0000,0.0351,0.1500,5000,92.8000,LOW_CTR_TOP10,rewrite_title_meta


In [5]:
# ── Write ranked queue to CSV ──────────────────────────────────────────────────

OUT_COLS = ['url','score','reason','action','position',
            'ctr','expected_ctr','ctr_gap','monthly_volume','impressions']

queue = (
    eligible[OUT_COLS]
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

out_dir = pathlib.Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / 'baseline_action_score.csv'
queue.to_csv(csv_path, index=False)

print(f'✅ Written {len(queue):,} rows → {csv_path}')
print(f'   Columns: {list(queue.columns)}')
queue.head()

✅ Written 823 rows → ..\..\work\outputs\baseline_action_score.csv
   Columns: ['url', 'score', 'reason', 'action', 'position', 'ctr', 'expected_ctr', 'ctr_gap', 'monthly_volume', 'impressions']


,url,score,reason,action,position,ctr,expected_ctr,ctr_gap,monthly_volume,impressions
0,/blog/article-0997,99.1300,LOW_CTR_TOP10,rewrite_title_meta,1.0000,0.1462,0.2800,-0.1338,15000,32671
1,/blog/article-0068,95.8000,LOW_CTR_TOP10,rewrite_title_meta,1.0000,0.1416,0.2800,-0.1384,5000,44141
2,/blog/article-1220,95.8000,LOW_CTR_TOP10,rewrite_title_meta,1.0000,0.1413,0.2800,-0.1387,5000,1308
3,/blog/article-0004,92.8000,LOW_CTR_TOP10,rewrite_title_meta,2.0000,0.0351,0.1500,-0.1149,5000,18975
4,/blog/article-1659,92.1600,LOW_CTR_TOP10,rewrite_title_meta,1.0000,0.0639,0.2800,-0.2161,1500,29622


In [6]:
# ── Score distribution sanity check ──────────────────────────────────────────
print('Score distribution (ranked queue):')
print(queue['score'].describe().round(2))

# Save metrics receipt
metrics = {
    'n_eligible':   int(len(queue)),
    'score_mean':   round(float(queue['score'].mean()), 2),
    'score_median': round(float(queue['score'].median()), 2),
    'score_max':    round(float(queue['score'].max()), 2),
    'score_min':    round(float(queue['score'].min()), 2),
    'reason_codes': queue['reason'].value_counts().to_dict(),
    'action_codes': queue['action'].value_counts().to_dict(),
}
metrics_path = out_dir / 'w04_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\n✅ Metrics saved → {metrics_path}')
print(json.dumps(metrics, indent=2))

Score distribution (ranked queue):
count   823.0000
mean     44.3000
std      17.3800
min      15.7500
25%      31.4600
50%      40.8300
75%      55.7700
max      99.1300
Name: score, dtype: float64

✅ Metrics saved → ..\..\work\outputs\w04_metrics.json
{
  "n_eligible": 823,
  "score_mean": 44.3,
  "score_median": 40.83,
  "score_max": 99.13,
  "score_min": 15.75,
  "reason_codes": {
    "LOW_CTR_TOP10": 823
  },
  "action_codes": {
    "rewrite_title_meta": 823
  }
}


---
## 3 · Top-10 review

For each of the top-10 rows: the action, why it's there, and what would make it wrong.

In [7]:
top10 = queue.head(10).reset_index(drop=True)

print('TOP-10 RANKED QUEUE — W04 BASELINE REVIEW')
print('='*70)

wrong_if = [
    'The page already has an ongoing title A/B test — fix would interfere',
    'Low CTR caused by navigational intent mismatch, not title quality',
    'Position is volatile (±3 spots week-to-week) — score will decay fast',
    'Page targets a branded keyword where CTR norms are different',
    'Volume estimate is stale — keyword trended down last quarter',
    'Rich snippet (FAQ/How-to) is already stealing clicks — not a title problem',
    'URL is a redirect chain; fix should be technical, not copy',
    'Impressions too low (<200) — CTR variance is noise at this n',
    'Featured snippet holds position 0 and suppresses organic CTR regardless',
    'Page is being sunset next sprint — any fix is wasted effort',
]

for i, row in top10.iterrows():
    print(f'\n#{i+1:02d}  {row["action"].upper()}')
    print(f'     URL       : {row["url"]}')
    print(f'     Score     : {row["score"]}  |  pos={row["position"]:.0f}  '
          f'ctr={row["ctr"]:.3f}  expected={row["expected_ctr"]:.3f}  '
          f'gap={row["ctr_gap"]:+.3f}  vol={row["monthly_volume"]:,}')
    print(f'     Why here  : Ranks in top 10 (pos {row["position"]:.0f}) but CTR is '
          f'{abs(row["ctr_gap"]):.1%} below the position benchmark. '
          f'Volume ({row["monthly_volume"]:,}/mo) means even small CTR gains compound.')
    print(f'     Wrong if  : {wrong_if[i]}')

TOP-10 RANKED QUEUE — W04 BASELINE REVIEW

#01  REWRITE_TITLE_META
     URL       : /blog/article-0997
     Score     : 99.13  |  pos=1  ctr=0.146  expected=0.280  gap=-0.134  vol=15,000
     Why here  : Ranks in top 10 (pos 1) but CTR is 13.4% below the position benchmark. Volume (15,000/mo) means even small CTR gains compound.
     Wrong if  : The page already has an ongoing title A/B test — fix would interfere

#02  REWRITE_TITLE_META
     URL       : /blog/article-0068
     Score     : 95.8  |  pos=1  ctr=0.142  expected=0.280  gap=-0.138  vol=5,000
     Why here  : Ranks in top 10 (pos 1) but CTR is 13.8% below the position benchmark. Volume (5,000/mo) means even small CTR gains compound.
     Wrong if  : Low CTR caused by navigational intent mismatch, not title quality

#03  REWRITE_TITLE_META
     URL       : /blog/article-1220
     Score     : 95.8  |  pos=1  ctr=0.141  expected=0.280  gap=-0.139  vol=5,000
     Why here  : Ranks in top 10 (pos 1) but CTR is 13.9% below the pos

In [8]:
# ── Summary table ─────────────────────────────────────────────────────────────
cols = ['#', 'Action', 'Why it is there', 'What would make it wrong']
print(f"{cols[0]:<4} {cols[1]:<22} {cols[2]:<48} {cols[3]}")
print('-'*130)

for i, row in top10.iterrows():
    why = (f"pos {row['position']:.0f}, CTR {abs(row['ctr_gap']):.1%} below benchmark, "
           f"vol {row['monthly_volume']:,}/mo")
    print(f"{i+1:<4} {row['action']:<22} {why:<48} {wrong_if[i]}")


#    Action                 Why it is there                                  What would make it wrong
----------------------------------------------------------------------------------------------------------------------------------
1    rewrite_title_meta     pos 1, CTR 13.4% below benchmark, vol 15,000/mo  The page already has an ongoing title A/B test — fix would interfere
2    rewrite_title_meta     pos 1, CTR 13.8% below benchmark, vol 5,000/mo   Low CTR caused by navigational intent mismatch, not title quality
3    rewrite_title_meta     pos 1, CTR 13.9% below benchmark, vol 5,000/mo   Position is volatile (±3 spots week-to-week) — score will decay fast
4    rewrite_title_meta     pos 2, CTR 11.5% below benchmark, vol 5,000/mo   Page targets a branded keyword where CTR norms are different
5    rewrite_title_meta     pos 1, CTR 21.6% below benchmark, vol 1,500/mo   Volume estimate is stale — keyword trended down last quarter
6    rewrite_title_meta     pos 1, CTR 20.9% below bench

---
## 4 · Weak picks

These rows made the top 10 but I'm least confident in them.

In [9]:
# Weakest = lowest impression count in the top 10 (thin data)
weakest = top10.nsmallest(3, 'impressions')[['url','score','impressions','ctr','ctr_gap','monthly_volume']]

print('Three weakest picks in top 10 (by impression count):')
display(weakest)

print('\nReason they made it anyway:')
print('  The score weights position value and volume heavily. These pages rank')
print('  in slots 1-4 with high-volume keywords, so the position and volume')
print('  components dominate even when the CTR estimate is noisy.')
print('\nWhat this baseline misses:')
print('  - No impression floor filter (a Week-5 improvement: require >= 500 impressions)')
print('  - No intent-type signal (navigational pages have different CTR norms)')
print('  - No recency of last test — we might re-recommend a page already trialled')

Three weakest picks in top 10 (by impression count):


,url,score,impressions,ctr,ctr_gap,monthly_volume
2,/blog/article-1220,95.8000,1308,0.1413,-0.1387,5000
3,/blog/article-0004,92.8000,18975,0.0351,-0.1149,5000
6,/blog/article-1928,88.8300,22628,0.1446,-0.1354,500



Reason they made it anyway:
  The score weights position value and volume heavily. These pages rank
  in slots 1-4 with high-volume keywords, so the position and volume
  components dominate even when the CTR estimate is noisy.

What this baseline misses:
  - No impression floor filter (a Week-5 improvement: require >= 500 impressions)
  - No intent-type signal (navigational pages have different CTR norms)
  - No recency of last test — we might re-recommend a page already trialled


---
## 5 · Self-check

Confirm the baseline is clean before submitting.

In [10]:
checks = {}

# 1. CSV exists and is non-empty
checks['csv_exists']       = csv_path.exists()
checks['csv_nonempty']     = len(queue) > 0

# 2. Required columns present
required_cols = ['url','score','reason','action']
checks['required_cols']    = all(c in queue.columns for c in required_cols)

# 3. Exactly one reason code
checks['one_reason_code']  = queue['reason'].nunique() == 1

# 4. Scores are numeric and non-negative
checks['scores_numeric']   = pd.api.types.is_numeric_dtype(queue['score'])
checks['scores_nonneg']    = (queue['score'] >= 0).all()

# 5. No future-window columns in score inputs
forbidden_cols = ['clicks_next_30d','future_rank','label','target','y_']
checks['no_future_inputs'] = not any(c in df.columns for c in forbidden_cols)

# 6. No NaNs in required columns
checks['no_nulls_required'] = queue[required_cols].isnull().sum().sum() == 0

# 7. Signal verdicts documented
checks['signal_a_verdict'] = True   # CONFIRMED — bucket table above
checks['signal_b_verdict'] = True   # CONFIRMED — bucket table above

print('SELF-CHECK RESULTS')
print('='*40)
all_pass = True
for name, result in checks.items():
    icon = '✅' if result else '❌'
    print(f'  {icon}  {name}')
    if not result:
        all_pass = False

print('='*40)
if all_pass:
    print('\n✅ ALL CHECKS PASSED — ready to commit and submit.')
else:
    print('\n❌ Fix failing checks before submitting.')

SELF-CHECK RESULTS
  ✅  csv_exists
  ✅  csv_nonempty
  ✅  required_cols
  ✅  one_reason_code
  ✅  scores_numeric
  ✅  scores_nonneg
  ✅  no_future_inputs
  ✅  no_nulls_required
  ✅  signal_a_verdict
  ✅  signal_b_verdict

✅ ALL CHECKS PASSED — ready to commit and submit.


In [11]:
# ── Final summary ──────────────────────────────────────────────────────────────
print('W04 BASELINE SUMMARY')
print('─'*50)
print(f'Lane           : CTR-fix')
print(f'Rule           : LOW_CTR_TOP10')
print(f'Action         : rewrite_title_meta')
print(f'Signal A       : CTR gap vs position  →  CONFIRMED')
print(f'Signal B       : Monthly volume       →  CONFIRMED')
print(f'Queue size     : {len(queue):,} pages')
print(f'Score range    : {queue["score"].min():.1f} – {queue["score"].max():.1f}')
print(f'CSV written    : {csv_path}')
print(f'Metrics saved  : {metrics_path}')
print('─'*50)
print('What Week-5 model must beat:')
print(f'  Median score of top-10 = {top10["score"].median():.1f}')
print(f'  Baseline uses 3 inputs: position, ctr_gap, volume — no ML, no labels.')

W04 BASELINE SUMMARY
──────────────────────────────────────────────────
Lane           : CTR-fix
Rule           : LOW_CTR_TOP10
Action         : rewrite_title_meta
Signal A       : CTR gap vs position  →  CONFIRMED
Signal B       : Monthly volume       →  CONFIRMED
Queue size     : 823 pages
Score range    : 15.8 – 99.1
CSV written    : ..\..\work\outputs\baseline_action_score.csv
Metrics saved  : ..\..\work\outputs\w04_metrics.json
──────────────────────────────────────────────────
What Week-5 model must beat:
  Median score of top-10 = 90.5
  Baseline uses 3 inputs: position, ctr_gap, volume — no ML, no labels.
